In [0]:
print("Spark version:", spark.version)

print("\nCurrent catalog:")
print(spark.sql("SELECT current_catalog()").first()[0])

print("\nCurrent schema:")
print(spark.sql("SELECT current_schema()").first()[0])

Spark version: 4.2.0

Current catalog:
workspace

Current schema:
default


In [0]:
spark.sql("""
CREATE CATALOG IF NOT EXISTS imdb_sentiment
""")

print("Catalog created or already exists.")

display(spark.sql("SHOW CATALOGS"))

Catalog created or already exists.


catalog
borewell_water_intelligence
buymore
dbacademy
dhl_logistic_intelligence
imdb_sentiment
samples
system
training
workspace


In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS imdb_sentiment.bronze
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS imdb_sentiment.silver
""")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS imdb_sentiment.gold
""")

print("Medallion schemas created successfully!")

display(spark.sql("""
SHOW SCHEMAS IN imdb_sentiment
"""))

Medallion schemas created successfully!


databaseName
bronze
default
gold
information_schema
silver


In [0]:
import urllib.request

url = "https://huggingface.co"

try:
    response = urllib.request.urlopen(url, timeout=10)
    print("Hugging Face status:", response.status)
    print("Internet access is available.")
except Exception as e:
    print("Could not access Hugging Face.")
    print("Error:", str(e))

Hugging Face status: 200
Internet access is available.


In [0]:
print("Spark version:", spark.version)

display(spark.sql("SELECT current_catalog(), current_schema()"))

Spark version: 4.2.0


current_catalog(),current_schema()
workspace,default


In [0]:
spark.sql("USE CATALOG imdb_sentiment")
spark.sql("USE SCHEMA bronze")

print("Current catalog:", spark.sql("SELECT current_catalog()").first()[0])
print("Current schema:", spark.sql("SELECT current_schema()").first()[0])

Current catalog: imdb_sentiment
Current schema: bronze


In [0]:
# Step 7 — Load the IMDb dataset into Bronze

try:
    import datasets
    print("Hugging Face datasets version:", datasets.__version__)
    print("Ready to load the IMDb dataset.")
except ImportError:
    print("Hugging Face datasets is not installed.")

Hugging Face datasets is not installed.


In [0]:
# Step 8 — Install the Hugging Face datasets library

%pip install datasets

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 13.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 855.2/855.2 kB 26.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 98.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 99.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 MB 181.0 MB/s  0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 21.0.0
    Not uninstalling pyarrow at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-40cd734e-8605-465c-b2d1-4ef753100f73
    Can't uninstall 'pyarrow'. No files were found to uninstall.
  Attempting uninstall: idna
    Found existing installation: idna 3.11
    Not uninstalling idna at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-40cd734e-8605-465c-b2d1-4ef753100f73
    Can't 

In [0]:
import datasets
import pyarrow

print("Datasets version:", datasets.__version__)
print("PyArrow version:", pyarrow.__version__)

print("Spark version:", spark.version)

display(spark.range(5))

Datasets version: 5.1.0
PyArrow version: 21.0.0
Spark version: 4.2.0


id
0
1
2
3
4


In [0]:
# Step 1 — Load the IMDb dataset

from datasets import load_dataset

imdb_dataset = load_dataset("stanfordnlp/imdb")

print("Dataset loaded successfully!")
print(imdb_dataset)

/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:56: UserWarning: The cache_dir for this dataset is /tmp/.hf.data.cache, which is not a persistent path.Therefore, if/when the cluster restarts, the downloaded dataset will be lost.The persistent storage options for this workspace/cluster config are: [UC Volumes].Please update either `cache_dir` or the environment variable `HF_DATASETS_CACHE`to be under one of the following root directories: ['/Volumes/']
  warnings.warn(warning_message)


README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

---------------------------------------------------------------------------
TypeError                                 Traceback (most recent call last)
File <command-4706448133912171>, line 5
      1 # Step 1 — Load the IMDb dataset
      3 from datasets import load_dataset
----> 5 imdb_dataset = load_dataset("stanfordnlp/imdb")
      7 print("Dataset loaded successfully!")
      8 print(imdb_dataset)

File /databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:157, in Patcher.__call__(self, *args, **kwargs)
    156 def __call__(self, *args, **kwargs):
--> 157     return self._fn(*args, **kwargs)

File /databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:180, in apply_load_datasets_patch.<locals>.hook.<locals>.load_dataset_wrapper.<locals>.patched_load_dataset(*args, **kwargs)
    178 if cache_dir:
    179     validate_storage_dir(conf, dbutils, cache_dir)
--> 180     _show_warning_if_cache_dir_is_local(datasets, conf, dbutils, *args, **kwargs)
    181

In [0]:
display(spark.sql("SHOW VOLUMES IN imdb_sentiment.bronze"))

database,volume_name


In [0]:
import datasets
import huggingface_hub
import pyarrow

print("datasets:", datasets.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("pyarrow:", pyarrow.__version__)
print("Spark:", spark.version)

datasets: 5.1.0
huggingface_hub: 2.3.0
pyarrow: 21.0.0
Spark: 4.2.0


In [0]:
%pip install "huggingface_hub==0.36.0"

Looking in indexes: [REDACTED]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.1/566.1 kB 14.6 MB/s  0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 2.3.0
    Uninstalling huggingface_hub-2.3.0:
      Successfully uninstalled huggingface_hub-2.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 5.1.0 requires huggingface-hub<3.0,>=1.31.0, but you have huggingface-hub 0.36.0 which is incompatible.
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install "datasets==3.3.2" "huggingface_hub==0.36.0"

Looking in indexes: [REDACTED]
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.19
    Uninstalling multiprocess-0.70.19:
      Successfully uninstalled multiprocess-0.70.19
  Attempting uninstall: datasets
    Found existing installation: datasets 5.1.0
    Uninstalling datasets-5.1.0:
      Successfully uninstalled datasets-5.1.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [datasets]
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import datasets
import huggingface_hub
import pyarrow

print("datasets:", datasets.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("pyarrow:", pyarrow.__version__)
print("Spark:", spark.version)

display(spark.range(5))

datasets: 3.3.2
huggingface_hub: 0.36.0
pyarrow: 26.0.0
Spark: 4.2.0


id
0
1
2
3
4


In [0]:
from datasets import load_dataset

imdb_dataset = load_dataset(
    "stanfordnlp/imdb",
    cache_dir="/tmp/.hf.data.cache"
)

print("Available splits:", list(imdb_dataset.keys()))

print("Training rows:", len(imdb_dataset["train"]))
print("Test rows:", len(imdb_dataset["test"]))

print("Training example:")
print(imdb_dataset["train"][0])

/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:56: UserWarning: The cache_dir for this dataset is /tmp/.hf.data.cache, which is not a persistent path.Therefore, if/when the cluster restarts, the downloaded dataset will be lost.The persistent storage options for this workspace/cluster config are: [UC Volumes].Please update either `cache_dir` or the environment variable `HF_DATASETS_CACHE`to be under one of the following root directories: ['/Volumes/']
  warnings.warn(warning_message)
/databricks/python_shell/lib/dbruntime/huggingface_patches/datasets.py:24: UserWarning: During large dataset downloads, there could be multiple progress bar widgets that can cause performance issues for your notebook or browser. To avoid these issues, use `datasets.utils.logging.disable_progress_bar()` to turn off the progress bars.
  warnings.warn(


plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

Available splits: ['train', 'test', 'unsupervised']
Training rows: 25000
Test rows: 25000
Training example:
{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW 

In [0]:
from pyspark.sql.functions import current_timestamp

# Convert the source splits to Spark DataFrames
train_spark_df = spark.createDataFrame(
    imdb_dataset["train"].to_pandas()
)

test_spark_df = spark.createDataFrame(
    imdb_dataset["test"].to_pandas()
)

# Add ingestion metadata
train_bronze_df = train_spark_df.withColumn(
    "ingested_at", current_timestamp()
)

test_bronze_df = test_spark_df.withColumn(
    "ingested_at", current_timestamp()
)

# Save as Delta tables in the Bronze schema
train_bronze_df.write.format("delta").mode("overwrite").saveAsTable(
    "imdb_sentiment.bronze.train_reviews"
)

test_bronze_df.write.format("delta").mode("overwrite").saveAsTable(
    "imdb_sentiment.bronze.test_reviews"
)

print("Bronze tables created successfully.")
print("Training rows:", spark.table("imdb_sentiment.bronze.train_reviews").count())
print("Test rows:", spark.table("imdb_sentiment.bronze.test_reviews").count())

Bronze tables created successfully.
Training rows: 25000
Test rows: 25000


In [0]:
from pyspark.sql import functions as F

bronze_train = spark.table("imdb_sentiment.bronze.train_reviews")

print("Schema:")
bronze_train.printSchema()

print("Label distribution:")
display(
    bronze_train.groupBy("label")
    .count()
    .orderBy("label")
)

print("Sample records:")
display(bronze_train.select("text", "label", "ingested_at").limit(5))

Schema:
root
 |-- text: string (nullable = true)
 |-- label: long (nullable = true)
 |-- ingested_at: timestamp (nullable = true)

Label distribution:


label,count
0,12500
1,12500


Sample records:


text,label,ingested_at
"Allen goes to the country (somewhere he hates going in real life) and has a weekend with his friends - which are the usual successful white middle-class bellyaching types that feature in many of his films.I usually find something to amuse in Woody Allen comedies, but here he really falls totally flat on his face. Even the one-liners seem to have deserted him. The really is no plot (bar bits and pieces of cod Shakespeare) - but Allen seems to use the location to allow a semi-mystical air, which just makes the thing even more witless and half-baked.It just doesn't work at any level and is just a giant bore. The best thing about this film (apart from the end credits coming up) is that the bad reviews seem to get him to wake up and realise that simply throwing together a slapdash script and casting your mates in it doesn't make for entertainment.",0,2026-10-29T10:58:27.967Z
"The greatest sin in life is being dull, and this movie is crashingly boring. its funny, its left out of his ""a life in film"" documentary. He goes from a long piece on ""Stardust Memories"" and then fast forwards to ""Zelig"". This little piece of cubic zirconia just isn't worth the effort.",0,2026-10-29T10:58:27.967Z
"Hooper is Not Funny, Not Fasted paced, Not romantic and Non informative. There is no real drama. You would think that a movie about the world's greatest stuntman would have some drama, there was an attempt but it didn't seem real. No Character study, no lessons learned, it did not even look like the actors were having any real fun, they were just trying to act like they were having fun. There is no reason to watch unless you like to look at Burt and want get an occasional glimpse of Sally. Prancer the horse was beautiful and did what he was supposed to do. In fact Prancer was the best actor in this movie. Smoky and the Bandit was such a fun movie that I was ready to like Hooper. This movie turned out to be a real disappointment and waste of time",0,2026-10-29T10:58:27.967Z
"""Smokey And The Bandit"" wasn't exactly Shakespeare, but then nobody wanted it to be. It was lowdown slapstick, but it did have brains. It had a very smart script with definable characters and a fun wrap-up. People came out of the theater smiling. ""Hooper"" provides none of this. There is no reason to smile. If it's supposed to be a tribute to the Hollywood Stuntman, it makes them look awfully lazy by providing nothing but badly-choreographed fight scenes and one of the most unconvincing car-jumps I've ever seen. It all looks phony, badly-filmed almost on purpose. Poor Sally Field (as the girlfriend who wrings her hands on the sidelines) is given her weakest role, with not a single funny or smart line (""If you do that jump, I won't be here when you get back""). Burt Reynolds keeps looking at the camera and winking, but the joke is on any audience who sits through ""Hooper"". * from ****",0,2026-10-29T10:58:27.967Z
"This is possibly the worst fencing, sword-fighting, movie ever made. That is not just because the so called sports fencing is poor but because the plot, characters are so weak that they've got to throw in a semi nude sex scene and, later, supposed group dancing around a fencing scene in the fencing club trying, I suppose to maintain audience interest. What a waste of F. Murray Abraham's talents. You're better served with overblown swashbuckling movies like Zorro, Scaramouche, anything that has Basil Rathbone as the villain. As a fencer myself I recognize the near impossibility of capturing fencing as a sport on film, but if it ever happens it's got to have fresher, better drawn characters and a plot with more depth.",0,2026-10-29T10:58:27.967Z
